In [6]:
#!/usr/bin/env python3
"""
Enhanced Multilingual Hate Speech Detection - FIXED VERSION
Building on your 60% F1 baseline with:
1. Bilingual input (original + English translation)
2. Smart paraphrasing-based upsampling 
3. Rich textual context from features
4. Careful incremental improvements
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, 
    get_linear_schedule_with_warmup, pipeline
)
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os
import json
import random
import warnings
from typing import Dict, List, Optional, Tuple
import re
warnings.filterwarnings('ignore')

# Configuration - keeping your working baseline
CSV_FILE = '../Datasets/labeled_comments.csv'
MODEL_NAME = "unitary/toxic-bert"  # Your working model
MAX_LENGTH = 384  # Increased for bilingual + context
BATCH_SIZE = 6  # Reduced due to longer sequences
GRADIENT_ACCUMULATION_STEPS = 3
NUM_TRAIN_EPOCHS = 10
LEARNING_RATE = 8e-6  # Slightly lower for stability
WARMUP_RATIO = 0.15
TEST_SIZE = 0.2
RANDOM_SEED = 42

# Enhanced rebalancing parameters
DOWNSAMPLE_NON_OFFENSIVE_RATIO = 0.2
MIN_SAMPLES_PER_CLASS = 800  # Increased due to better upsampling
MAX_NON_OFFENSIVE_SAMPLES = 3000
PARAPHRASE_RATIO = 0.4  # How many hate comments to paraphrase

# Translation and paraphrasing models
TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-mul-en"  # Multilingual to English
PARAPHRASE_MODEL = "Vamsi/T5_Paraphrase_Paws"  # T5 for paraphrasing

def set_seeds(seed=RANDOM_SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seeds()

# JSON serialization helper function
def convert_to_json_serializable(obj):
    """Convert numpy/pandas types to JSON serializable types"""
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_serializable(v) for v in obj]
    else:
        return obj

class TextProcessor:
    """Handles translation and paraphrasing"""
    
    def __init__(self):
        print("Loading translation and paraphrasing models...")
        device = 0 if torch.cuda.is_available() else -1
        
        # Translation pipeline
        try:
            self.translator = pipeline(
                "translation", 
                model=TRANSLATION_MODEL, 
                device=device,
                max_length=256
            )
        except:
            print("Warning: Translation model failed to load, using identity function")
            self.translator = None
        
        # Paraphrasing pipeline  
        try:
            self.paraphraser = pipeline(
                "text2text-generation",
                model=PARAPHRASE_MODEL,
                device=device,
                max_length=256
            )
        except:
            print("Warning: Paraphrasing model failed to load, using simple transforms")
            self.paraphraser = None
    
    def translate_to_english(self, text: str, source_lang: str = None) -> str:
        """Translate text to English if not already English"""
        if not self.translator or not text.strip():
            return text
            
        try:
            # Simple language detection
            if source_lang == 'en' or self._is_likely_english(text):
                return text
                
            result = self.translator(text, max_length=256, truncation=True)
            if result and len(result) > 0:
                return result[0]['translation_text']
        except:
            pass
        return text
    
    def _is_likely_english(self, text: str) -> bool:
        """Simple heuristic for English detection"""
        english_words = {'the', 'and', 'is', 'are', 'was', 'were', 'have', 'has', 'this', 'that'}
        words = text.lower().split()[:10]  # Check first 10 words
        english_count = sum(1 for word in words if word in english_words)
        return english_count >= 2 or len([c for c in text if ord(c) < 128]) / len(text) > 0.8
    
    def paraphrase_text(self, text: str) -> str:
        """Generate paraphrase of text for data augmentation"""
        if not self.paraphraser or not text.strip():
            return self._simple_paraphrase(text)
        
        try:
            # Prepare input for T5 paraphrasing
            input_text = f"paraphrase: {text}"
            result = self.paraphraser(
                input_text, 
                max_length=256, 
                num_return_sequences=1,
                temperature=0.8,
                do_sample=True,
                truncation=True
            )
            
            if result and len(result) > 0:
                paraphrase = result[0]['generated_text'].strip()
                # Basic quality check
                if len(paraphrase) > 10 and paraphrase != text:
                    return paraphrase
        except:
            pass
        
        return self._simple_paraphrase(text)
    
    def _simple_paraphrase(self, text: str) -> str:
        """Fallback simple paraphrasing"""
        # Simple transformations that preserve hate speech patterns
        transformations = [
            (r'\bso\b', 'very'),
            (r'\breally\b', 'extremely'), 
            (r'\bawesome\b', 'amazing'),
            (r'\bterrible\b', 'awful'),
            (r'\bstupid\b', 'dumb'),
            (r'\!+', '!'),
            (r'\.+', '.'),
        ]
        
        result = text
        for pattern, replacement in transformations:
            if random.random() < 0.3:  # Apply randomly
                result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
        
        # Add/remove punctuation occasionally
        if random.random() < 0.2:
            result = result.rstrip('!.') + '.'
        
        return result if result != text else text

def create_rich_context(row: pd.Series, additional_features: List[str]) -> str:
    """Create rich textual context from additional features"""
    context_parts = []
    
    # Language context
    lang_features = [f for f in additional_features if f.startswith('Language_') and row.get(f) == 1]
    if lang_features:
        lang = lang_features[0].replace('Language_', '').replace('_', '-')
        if lang != 'en':
            context_parts.append(f"language:{lang}")
    
    # Gender and sport context
    if row.get('Gender_of_Sport_Female') == 1:
        context_parts.append("women's football")
    elif row.get('Gender_of_Sport_Male') == 1:
        context_parts.append("men's football")
    
    # Club information
    club_features = [f for f in additional_features if f.startswith('Club_') and row.get(f) == 1 and f != 'Club Region_England']
    if club_features:
        club = club_features[0].replace('Club_', '').replace('_', ' ')
        context_parts.append(f"about:{club}")
    
    # Region information
    region_features = [f for f in additional_features if f.startswith('Club_Region_') and row.get(f) == 1]
    if region_features:
        region = region_features[0].replace('Club_Region_', '')
        context_parts.append(f"region:{region}")
    
    # Channel structure
    if row.get('Channel_Structure_Unified') == 1:
        context_parts.append("unified channel")
    elif row.get('Channel_Structure_Seperate') == 1:
        context_parts.append("separate channel")
    
    # Commenter gender
    if row.get('Commenter_Gender_male') == 1:
        context_parts.append("male commenter")
    
    # Popularity context
    video_pop = row.get('Video_Popularity', 0)
    if pd.notna(video_pop):
        if video_pop > 1000000:
            context_parts.append("viral video")
        elif video_pop > 100000:
            context_parts.append("popular video")
    
    comment_pop = row.get('Comment_Popularity', 0) 
    if pd.notna(comment_pop) and comment_pop > 100:
        context_parts.append("popular comment")
    
    return " | ".join(context_parts) if context_parts else ""

def enhanced_rebalancing_with_paraphrasing(
    df: pd.DataFrame, 
    hate_labels: List[str], 
    non_offensive_col: str,
    processor: TextProcessor,
    additional_features: List[str]
) -> pd.DataFrame:
    """Enhanced rebalancing with intelligent paraphrasing"""
    print("\n=== ENHANCED REBALANCING WITH PARAPHRASING ===")
    
    hate_samples = []
    
    for label in hate_labels:
        label_samples = df[df[label] == 1].copy()
        
        if len(label_samples) == 0:
            # Create synthetic samples as before
            other_hate_mask = df[hate_labels].sum(axis=1) > 0
            if other_hate_mask.sum() > 0:
                n_synthetic = min(100, other_hate_mask.sum())
                candidate_samples = df[other_hate_mask].sample(
                    n=n_synthetic, random_state=RANDOM_SEED + hash(label) % 1000
                ).copy()
                candidate_samples[label] = 1
                label_samples = candidate_samples
                print(f"  Created {len(label_samples)} synthetic samples for {label}")
        
        if len(label_samples) > 0:
            target_samples = max(MIN_SAMPLES_PER_CLASS, len(label_samples))
            
            if len(label_samples) < target_samples:
                needed = target_samples - len(label_samples)
                upsampled_rows = []
                
                # Determine how many to paraphrase vs simple augment
                n_paraphrase = int(needed * PARAPHRASE_RATIO)
                n_simple = needed - n_paraphrase
                
                print(f"  Upsampling {label}: {len(label_samples)} -> {target_samples}")
                print(f"    Using paraphrasing: {n_paraphrase}, simple augment: {n_simple}")
                
                # Paraphrasing-based augmentation
                for i in tqdm(range(n_paraphrase), desc=f"Paraphrasing {label}"):
                    base_sample = label_samples.sample(n=1, random_state=RANDOM_SEED + i).iloc[0].copy()
                    original_comment = str(base_sample['Comment'])
                    
                    # Get language for context
                    lang_features = [f for f in additional_features if f.startswith('Language_') and base_sample.get(f) == 1]
                    source_lang = lang_features[0].replace('Language_', '') if lang_features else 'unknown'
                    
                    # Paraphrase the comment
                    paraphrased = processor.paraphrase_text(original_comment)
                    base_sample['Comment'] = paraphrased
                    base_sample['augmentation_method'] = 'paraphrase'
                    
                    upsampled_rows.append(base_sample)
                
                # Simple augmentation for the rest
                for i in range(n_simple):
                    base_sample = label_samples.sample(n=1, random_state=RANDOM_SEED + n_paraphrase + i).iloc[0].copy()
                    comment = str(base_sample['Comment'])
                    
                    # Simple transformations
                    if np.random.random() < 0.3:
                        comment = comment.strip()
                    if np.random.random() < 0.2:
                        comment = comment.replace('!', '.')
                    if np.random.random() < 0.1:
                        comment = comment.replace('  ', ' ')
                    
                    base_sample['Comment'] = comment
                    base_sample['augmentation_method'] = 'simple'
                    upsampled_rows.append(base_sample)
                
                if upsampled_rows:
                    upsampled_df = pd.DataFrame(upsampled_rows)
                    label_samples = pd.concat([label_samples, upsampled_df], ignore_index=True)
            
            hate_samples.append(label_samples)
    
    # Handle non-offensive samples (no paraphrasing needed)
    non_offensive_samples = df[df[non_offensive_col] == 1].copy()
    total_hate = sum(len(samples) for samples in hate_samples)
    
    target_non_offensive = min(MAX_NON_OFFENSIVE_SAMPLES, int(total_hate * 1.2))
    
    if len(non_offensive_samples) > target_non_offensive:
        non_offensive_sampled = non_offensive_samples.sample(
            n=target_non_offensive, random_state=RANDOM_SEED
        )
    else:
        non_offensive_sampled = non_offensive_samples
    
    print(f"  Non-offensive: {len(non_offensive_samples)} -> {len(non_offensive_sampled)}")
    
    # Combine all samples
    all_samples = [non_offensive_sampled] + hate_samples
    balanced_df = pd.concat(all_samples, ignore_index=True)
    balanced_df = balanced_df.drop_duplicates(subset=['Comment'], keep='first')
    
    print(f"\nBalanced dataset: {len(balanced_df)} samples")
    print("New distribution:")
    for col in ['Sexism', 'Racism', 'Vulgarity', 'Appearance', 'Ability', 'Non_offensive']:
        if col in balanced_df.columns:
            count = balanced_df[col].sum()
            pct = balanced_df[col].mean() * 100
            print(f"  {col}: {count} samples ({pct:.2f}%)")
    
    return balanced_df

class EnhancedMultilingualDataset(Dataset):
    """Enhanced dataset with bilingual input and rich context"""
    
    def __init__(self, tokenizer, dataframe, label_columns, additional_features, 
                 max_length, processor, augment=False):
        self.tokenizer = tokenizer
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_columns = label_columns
        self.additional_features = additional_features
        self.max_length = max_length
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        original_comment = str(row['Comment']) if 'Comment' in row.index else ""
        original_comment = original_comment.replace('\n', ' ').replace('\r', ' ').strip()
        
        # Get language information
        lang_features = [f for f in self.additional_features if f.startswith('Language_') and row.get(f) == 1]
        source_lang = lang_features[0].replace('Language_', '') if lang_features else 'unknown'
        
        # Create bilingual input
        if source_lang != 'en' and source_lang != 'unknown':
            # Translate to English
            english_translation = self.processor.translate_to_english(original_comment, source_lang)
            if english_translation != original_comment:
                bilingual_text = f"Original: {original_comment} | English: {english_translation}"
            else:
                bilingual_text = original_comment
        else:
            bilingual_text = original_comment
        
        # Create rich context
        context = create_rich_context(row, self.additional_features)
        
        # Combine everything
        if context:
            full_text = f"{bilingual_text} [CONTEXT] {context}"
        else:
            full_text = bilingual_text
        
        # Training-time augmentation
        if self.augment and np.random.random() < 0.05:
            full_text = full_text.strip()
        
        # Tokenize
        encoding = self.tokenizer(
            full_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        labels = torch.tensor([
            float(row[col]) if col in row.index else 0.0 
            for col in self.label_columns
        ], dtype=torch.float)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': labels
        }

# Main execution
print("=== ENHANCED MULTILINGUAL HATE SPEECH DETECTION ===")
print("Building on your 60% F1 baseline with advanced features")

# Load and prepare data
df = pd.read_csv(CSV_FILE)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df.columns = [c.replace('-', '_').replace(' ', '_') for c in df.columns]

LABEL_COLUMNS = ['Sexism', 'Racism', 'Vulgarity', 'Appearance', 'Ability', 'Non_offensive']
HATE_SPEECH_LABELS = [col for col in LABEL_COLUMNS if col != 'Non_offensive']

if 'Violence' in df.columns and 'Vulgarity' not in df.columns:
    df['Vulgarity'] = df['Violence']
    df = df.drop(columns=['Violence'])

# Additional features (same as your original)
ADDITIONAL_FEATURES = [f.replace('-', '_').replace(' ', '_') for f in [
    'Video Popularity', 'Comment Popularity', 'Gender of Sport_Female', 'Gender of Sport_Male',
    'Club_Arsenal', 'Club_Bayern München', 'Club_Chelsea', 'Club_Club América', 'Club_Corinthians',
    'Club_Juventus', 'Club_Manchester United', 'Club_Napoli', 'Club_Olympique Lyon',
    'Club_Paris FC', 'Club_Tottenham Hotspur', 'Club_Wolfsburg', 'Club Region_England',
    'Club Region_France', 'Club Region_Germany', 'Club Region_Italy', 'Club Region_Latin America',
    'Language_af', 'Language_ar', 'Language_bg', 'Language_bn', 'Language_ca', 'Language_cs',
    'Language_cy', 'Language_da', 'Language_de', 'Language_el', 'Language_en', 'Language_es',
    'Language_et', 'Language_fa', 'Language_fi', 'Language_fr', 'Language_he', 'Language_hi',
    'Language_hr', 'Language_hu', 'Language_id', 'Language_it', 'Language_ja', 'Language_ko',
    'Language_lt', 'Language_lv', 'Language_mk', 'Language_ml', 'Language_nl', 'Language_no',
    'Language_pl', 'Language_pt', 'Language_ro', 'Language_ru', 'Language_sk', 'Language_sl',
    'Language_so', 'Language_sq', 'Language_sv', 'Language_sw', 'Language_th', 'Language_tl',
    'Language_tr', 'Language_uk', 'Language_unknown', 'Language_ur', 'Language_vi',
    'Language_zh-cn', 'Language_zh-tw', 'Channel Structure_Seperate', 'Channel Structure_Unified',
    'Commenter Gender_male', 'Commenter Gender_unknown'
]]
ADDITIONAL_FEATURES = [f for f in ADDITIONAL_FEATURES if f in df.columns]

# Validate label columns
for col in LABEL_COLUMNS:
    if col not in df.columns:
        raise ValueError(f"Label column '{col}' not found")
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("\nOriginal class distribution:")
for col in LABEL_COLUMNS:
    count = df[col].sum()
    pct = df[col].mean() * 100
    print(f"  {col}: {count} samples ({pct:.2f}%)")

# Initialize text processor
processor = TextProcessor()

# Apply enhanced rebalancing with paraphrasing
df_balanced = enhanced_rebalancing_with_paraphrasing(
    df, HATE_SPEECH_LABELS, 'Non_offensive', processor, ADDITIONAL_FEATURES
)

# Use your original stratified split function
def improved_stratified_split(df, hate_labels, test_size=0.2):
    """Your original stratified split that worked"""
    print("\n=== STRATIFIED SPLIT ===")
    
    df['label_combination'] = df[LABEL_COLUMNS].apply(
        lambda x: ''.join([str(int(val)) for val in x]), axis=1
    )
    
    train_indices = []
    val_indices = []
    
    for label in hate_labels:
        label_indices = df[df[label] == 1].index.tolist()
        
        if len(label_indices) >= 4:
            n_val = max(2, int(len(label_indices) * test_size))
            val_sample = np.random.choice(label_indices, size=n_val, replace=False)
            train_sample = [idx for idx in label_indices if idx not in val_sample]
            
            train_indices.extend(train_sample)
            val_indices.extend(val_sample)
            print(f"  {label}: {len(train_sample)} train, {len(val_sample)} val")
        
        elif len(label_indices) >= 2:
            val_sample = np.random.choice(label_indices, size=1, replace=False)
            train_sample = [idx for idx in label_indices if idx not in val_sample]
            
            train_indices.extend(train_sample)
            val_indices.extend(val_sample)
            print(f"  {label}: {len(train_sample)} train, {len(val_sample)} val")
        
        elif len(label_indices) == 1:
            train_indices.extend(label_indices)
            print(f"  {label}: {len(label_indices)} train, 0 val (will be handled)")
    
    remaining_indices = [idx for idx in df.index if idx not in train_indices and idx not in val_indices]
    
    if remaining_indices:
        remaining_df = df.loc[remaining_indices]
        if len(remaining_df) > 0:
            train_rem, val_rem = train_test_split(
                remaining_df, test_size=test_size, random_state=RANDOM_SEED,
                stratify=remaining_df['Non_offensive'] if 'Non_offensive' in remaining_df.columns else None
            )
            train_indices.extend(train_rem.index.tolist())
            val_indices.extend(val_rem.index.tolist())
    
    train_df = df.loc[train_indices].reset_index(drop=True)
    val_df = df.loc[val_indices].reset_index(drop=True)
    
    # Ensure validation coverage
    print("\nValidation set verification:")
    for col in LABEL_COLUMNS:
        val_count = val_df[col].sum()
        train_count = train_df[col].sum()
        
        if val_count == 0 and col in hate_labels and train_count > 1:
            sample_to_move = train_df[train_df[col] == 1].sample(n=1, random_state=RANDOM_SEED)
            train_df = train_df.drop(sample_to_move.index).reset_index(drop=True)
            val_df = pd.concat([val_df, sample_to_move], ignore_index=True)
            val_count = 1
            train_count -= 1
            print(f"  {col}: Moved 1 sample to validation")
        
        print(f"  {col}: {train_count} train, {val_count} val")
    
    return train_df, val_df

train_df, val_df = improved_stratified_split(df_balanced, HATE_SPEECH_LABELS, TEST_SIZE)
print(f"\nFinal split - Train: {len(train_df)}, Validation: {len(val_df)}")

# Model setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification"
)
model.to(device)

# Create enhanced datasets
train_dataset = EnhancedMultilingualDataset(
    tokenizer, train_df, LABEL_COLUMNS, ADDITIONAL_FEATURES, MAX_LENGTH, processor, augment=True
)
val_dataset = EnhancedMultilingualDataset(
    tokenizer, val_df, LABEL_COLUMNS, ADDITIONAL_FEATURES, MAX_LENGTH, processor, augment=False
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Enhanced class weights (keeping your approach but refined)
def compute_enhanced_weights(train_df, label_columns):
    pos_weights = []
    
    print("\nComputing enhanced class weights:")
    for col in label_columns:
        pos_count = train_df[col].sum()
        neg_count = len(train_df) - pos_count
        
        if pos_count > 0:
            base_weight = neg_count / pos_count
            
            if col in HATE_SPEECH_LABELS:
                if pos_count < 300:
                    weight = min(base_weight * 2.2, 80.0)  # Slightly less aggressive
                elif pos_count < 600:
                    weight = min(base_weight * 1.8, 40.0)
                else:
                    weight = min(base_weight * 1.4, 20.0)
            else:
                weight = min(base_weight, 4.0)
        else:
            weight = 1.0
            
        pos_weights.append(weight)
        print(f"  {col}: pos={pos_count}, weight={weight:.2f}")
    
    return torch.tensor(pos_weights, dtype=torch.float).to(device)

pos_weights = compute_enhanced_weights(train_df, LABEL_COLUMNS)

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

total_steps = len(train_dataloader) * NUM_TRAIN_EPOCHS // GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# Evaluation function (keeping your working approach)
def evaluate_with_optimal_thresholds(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(outputs.logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    optimal_thresholds = []
    f1_scores_per_class = []
    
    for i in range(len(LABEL_COLUMNS)):
        if np.sum(all_labels[:, i]) > 0:
            precision, recall, thresholds = precision_recall_curve(all_labels[:, i], all_preds[:, i])
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
            best_idx = np.argmax(f1_scores)
            optimal_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            optimal_thresholds.append(optimal_threshold)
            f1_scores_per_class.append(f1_scores[best_idx])
        else:
            optimal_thresholds.append(0.5)
            f1_scores_per_class.append(0.0)
    
    binary_preds = np.zeros_like(all_preds)
    for i, threshold in enumerate(optimal_thresholds):
        binary_preds[:, i] = (all_preds[:, i] >= threshold).astype(int)
    
    f1_macro = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
    f1_micro = f1_score(all_labels, binary_preds, average='micro', zero_division=0)
    
    return {
        'loss': total_loss / len(dataloader),
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'optimal_thresholds': optimal_thresholds,
        'f1_per_class': f1_scores_per_class,
        'predictions': binary_preds,
        'labels': all_labels,
        'probabilities': all_preds
    }

# Training loop
print("\n=== ENHANCED MULTILINGUAL TRAINING ===")
best_f1_macro = 0
best_model_state = None
best_thresholds = None
patience = 5  # Slightly more patience for complex model
patience_counter = 0

for epoch in range(NUM_TRAIN_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_TRAIN_EPOCHS}")
    
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Training")
    
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        
        loss.backward()
        
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        progress_bar.set_postfix({'loss': f'{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}'})
    
    avg_train_loss = total_loss / len(train_dataloader)
    
    # Evaluate
    eval_results = evaluate_with_optimal_thresholds(model, val_dataloader, device)
    
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {eval_results['loss']:.4f}")
    print(f"Validation F1 Macro: {eval_results['f1_macro']:.4f}")
    print(f"Validation F1 Micro: {eval_results['f1_micro']:.4f}")
    
    print("Per-class F1:")
    for i, col in enumerate(LABEL_COLUMNS):
        print(f"  {col}: {eval_results['f1_per_class'][i]:.4f}")
    
    # Save best model
    if eval_results['f1_macro'] > best_f1_macro:
        best_f1_macro = eval_results['f1_macro']
        best_model_state = model.state_dict().copy()
        best_thresholds = eval_results['optimal_thresholds'].copy()
        patience_counter = 0
        print(f"*** NEW BEST F1 MACRO: {best_f1_macro:.4f} ***")
        
        # Save intermediate best model
        temp_save_path = "./temp_best_multilingual_model"
        os.makedirs(temp_save_path, exist_ok=True)
        torch.save(best_model_state, os.path.join(temp_save_path, 'model_state.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping after {patience} epochs without improvement")
            break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model with F1 macro: {best_f1_macro:.4f}")

print("\n" + "="*70)
print("ENHANCED MULTILINGUAL MODEL - FINAL RESULTS")
print("="*70)

final_results = evaluate_with_optimal_thresholds(model, val_dataloader, device)

print(f"\nFINAL ENHANCED RESULTS:")
print(f"F1 Macro: {final_results['f1_macro']:.4f}")
print(f"F1 Micro: {final_results['f1_micro']:.4f}")

print(f"\nBaseline Comparison:")
print(f"Original V3 Baseline: 60.0% F1 macro")
print(f"Enhanced Multilingual: {final_results['f1_macro']:.1%} F1 macro")
improvement = (final_results['f1_macro'] - 0.60) * 100
print(f"Improvement: {improvement:+.1f} percentage points")

print(f"\nOptimal thresholds:")
for i, col in enumerate(LABEL_COLUMNS):
    print(f"  {col}: {final_results['optimal_thresholds'][i]:.3f}")

print(f"\nPer-class F1 scores:")
for i, col in enumerate(LABEL_COLUMNS):
    print(f"  {col}: {final_results['f1_per_class'][i]:.4f}")

print(f"\nDetailed Classification Report:")
print(classification_report(
    final_results['labels'], 
    final_results['predictions'],
    target_names=LABEL_COLUMNS,
    zero_division=0
))

# Analyze enhancement contributions
print(f"\n" + "="*50)
print("ENHANCEMENT ANALYSIS")
print("="*50)

# Check translation usage
train_lang_dist = {}
for col in [c for c in ADDITIONAL_FEATURES if c.startswith('Language_')]:
    if col in train_df.columns:
        count = int(train_df[col].sum())  # Convert to native int
        if count > 0:
            lang = col.replace('Language_', '')
            train_lang_dist[lang] = count

print("\nLanguage distribution in training set:")
for lang, count in sorted(train_lang_dist.items(), key=lambda x: x[1], reverse=True):
    if count > 10:  # Only show languages with meaningful representation
        pct = count / len(train_df) * 100
        print(f"  {lang}: {count} samples ({pct:.1f}%)")

non_english_ratio = sum(count for lang, count in train_lang_dist.items() if lang != 'en') / len(train_df)
print(f"\nNon-English content: {non_english_ratio:.1%} of training data")

# Check augmentation distribution
if 'augmentation_method' in train_df.columns:
    aug_counts = train_df['augmentation_method'].value_counts()
    print(f"\nData augmentation breakdown:")
    for method, count in aug_counts.items():
        if pd.notna(method):
            print(f"  {method}: {count} samples")

# Save enhanced model
model_save_path = "./enhanced_multilingual_model"
os.makedirs(model_save_path, exist_ok=True)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Save comprehensive metadata with JSON serialization fix
metadata = {
    'model_type': 'enhanced_multilingual',
    'base_model': MODEL_NAME,
    'enhancements': [
        'bilingual_input_original_plus_english',
        'paraphrasing_based_upsampling',
        'rich_textual_context_from_features',
        'optimized_class_weights'
    ],
    'label_mapping': {i: label for i, label in enumerate(LABEL_COLUMNS)},
    'optimal_thresholds': [float(t) for t in final_results['optimal_thresholds']],
    'performance': {
        'f1_macro': float(final_results['f1_macro']),
        'f1_micro': float(final_results['f1_micro']),
        'baseline_f1_macro': 0.60,
        'improvement_percentage_points': float((final_results['f1_macro'] - 0.60) * 100)
    },
    'f1_per_class': {col: float(f1) for col, f1 in zip(LABEL_COLUMNS, final_results['f1_per_class'])},
    'training_config': {
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs_trained': epoch + 1 if 'epoch' in locals() else NUM_TRAIN_EPOCHS,
        'paraphrase_ratio': PARAPHRASE_RATIO,
        'min_samples_per_class': MIN_SAMPLES_PER_CLASS
    },
    'language_distribution': convert_to_json_serializable(train_lang_dist),
    'non_english_ratio': float(non_english_ratio),
    'dataset_stats': {
        'total_training_samples': len(train_df),
        'total_validation_samples': len(val_df),
        'total_features': len(ADDITIONAL_FEATURES)
    }
}

# Apply JSON serialization conversion
metadata = convert_to_json_serializable(metadata)

with open(os.path.join(model_save_path, 'enhanced_model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nEnhanced model saved to {model_save_path}")

# Cleanup temporary files
import shutil
if os.path.exists("./temp_best_multilingual_model"):
    shutil.rmtree("./temp_best_multilingual_model")

# Final summary and recommendations
print(f"\n" + "="*60)
print("ENHANCEMENT SUMMARY & NEXT STEPS")
print("="*60)

if final_results['f1_macro'] > 0.60:
    print("✅ SUCCESS: Enhanced model outperforms baseline!")
    print(f"   Improvement: {improvement:+.1f} percentage points")
    
    print(f"\n🚀 WHAT WORKED WELL:")
    print(f"   • Bilingual input (original + English translation)")
    print(f"   • Paraphrasing-based upsampling for hate comments") 
    print(f"   • Rich contextual features in natural language")
    print(f"   • Optimized training parameters")
    
    print(f"\n📈 FURTHER IMPROVEMENTS TO TRY:")
    print(f"   • Ensemble with RoBERTa or XLM-R for multilingual tasks")
    print(f"   • Back-translation for additional data augmentation")
    print(f"   • Cross-lingual consistency loss")
    print(f"   • Fine-tune translation model on domain-specific data")
    print(f"   • Experiment with different paraphrasing models")
    
    if final_results['f1_macro'] > 0.65:
        print(f"   🎯 EXCELLENT RESULT! Consider submitting to competition")
        
elif final_results['f1_macro'] > 0.58:
    print("⚠️  MARGINAL IMPROVEMENT: Close to baseline")
    print(f"   Try reducing complexity or adjusting hyperparameters")
    
    print(f"\n🔧 DEBUG SUGGESTIONS:")
    print(f"   • Check if translation quality is good")
    print(f"   • Verify paraphrasing preserves hate speech patterns") 
    print(f"   • Reduce MAX_LENGTH if memory issues")
    print(f"   • Try simpler context formatting")
    
else:
    print("❌ REGRESSION: Model performs worse than baseline")
    print(f"   Revert to baseline and make smaller incremental changes")
    
    print(f"\n🚨 IMMEDIATE ACTIONS:")
    print(f"   • Check data preprocessing pipeline")
    print(f"   • Verify translation/paraphrasing isn't corrupting data")
    print(f"   • Reduce model complexity")
    print(f"   • Use original simple context approach")

print(f"\n📊 Key Metrics:")
print(f"   • Languages processed: {len([l for l, c in train_lang_dist.items() if c > 0])}")
print(f"   • Non-English content: {non_english_ratio:.1%}")
print(f"   • Training samples: {len(train_df):,}")
print(f"   • Context length: {MAX_LENGTH} tokens")

if __name__ == "__main__":
    print(f"\n🎯 FINAL SCORE: {final_results['f1_macro']:.1%} F1 Macro")
    if final_results['f1_macro'] >= 0.62:
        print("🏆 ENHANCEMENT SUCCESSFUL!")
    elif final_results['f1_macro'] >= 0.58:
        print("📈 PROGRESS MADE - KEEP ITERATING!")
    else:
        print("🔄 BACK TO DRAWING BOARD - TRY SIMPLER ENHANCEMENTS")

=== ENHANCED MULTILINGUAL HATE SPEECH DETECTION ===
Building on your 60% F1 baseline with advanced features

Original class distribution:
  Sexism: 11 samples (1.29%)
  Racism: 3 samples (0.35%)
  Vulgarity: 12 samples (1.41%)
  Appearance: 8 samples (0.94%)
  Ability: 9 samples (1.06%)
  Non_offensive: 811 samples (95.41%)
Loading translation and paraphrasing models...

=== ENHANCED REBALANCING WITH PARAPHRASING ===
  Upsampling Sexism: 11 -> 800
    Using paraphrasing: 315, simple augment: 474


Paraphrasing Sexism: 100%|██████████| 315/315 [00:00<00:00, 483.41it/s]


  Upsampling Racism: 3 -> 800
    Using paraphrasing: 318, simple augment: 479


Paraphrasing Racism: 100%|██████████| 318/318 [00:00<00:00, 570.00it/s]


  Upsampling Vulgarity: 12 -> 800
    Using paraphrasing: 315, simple augment: 473


...

Training Loss: 0.3218
Validation Loss: 0.7030
Validation F1 Macro: 0.8735
Validation F1 Micro: 0.9514
Per-class F1:
  Sexism: 0.9565
  Racism: 0.6667
  Vulgarity: 0.8571
  Appearance: 1.0000
  Ability: 0.8000
  Non_offensive: 0.9607
*** NEW BEST F1 MACRO: 0.8735 ***

Epoch 10/10


Evaluating: 100%|██████████| 30/30 [01:40<00:00,  3.35s/it]


Training Loss: 0.2823
Validation Loss: 0.7181
Validation F1 Macro: 0.8735
Validation F1 Micro: 0.9514
Per-class F1:
  Sexism: 0.9565
  Racism: 0.6667
  Vulgarity: 0.8571
  Appearance: 1.0000
  Ability: 0.8000
  Non_offensive: 0.9607

Loaded best model with F1 macro: 0.8735

ENHANCED MULTILINGUAL MODEL - FINAL RESULTS


Evaluating: 100%|██████████| 30/30 [01:38<00:00,  3.27s/it]



FINAL ENHANCED RESULTS:
F1 Macro: 0.8735
F1 Micro: 0.9514

Baseline Comparison:
Original V3 Baseline: 60.0% F1 macro
Enhanced Multilingual: 87.4% F1 macro
Improvement: +27.4 percentage points

Optimal thresholds:
  Sexism: 0.968
  Racism: 0.932
  Vulgarity: 0.976
  Appearance: 0.959
  Ability: 0.917
  Non_offensive: 0.021

Per-class F1 scores:
  Sexism: 0.9565
  Racism: 0.6667
  Vulgarity: 0.8571
  Appearance: 1.0000
  Ability: 0.8000
  Non_offensive: 0.9607

Detailed Classification Report:
               precision    recall  f1-score   support

       Sexism       0.92      1.00      0.96        11
       Racism       1.00      0.50      0.67         2
    Vulgarity       0.86      0.86      0.86         7
   Appearance       1.00      1.00      1.00         5
      Ability       1.00      0.67      0.80         6
Non_offensive       0.92      1.00      0.96       159

    micro avg       0.93      0.98      0.95       190
    macro avg       0.95      0.84      0.87       190
 weigh